# 12. Functions as a Service: Fission

← [Jupyter](11-jupyter.ipynb) · next: [MPI](13-mpi.ipynb)

Sometimes the unit of work is smaller than an application. A function that
subsets a dataset when someone asks, a webhook receiver, a small lookup API, a
piece of glue between two systems — each is a handful of lines of Python, and
wrapping each one in an image, a chart and a Deployment that idles all day is
out of proportion.

CIRRUS runs **[Fission](https://fission.io/docs/)** for this. You write one
function; Fission keeps a small pool of warm runtime pods, and when an HTTP
request arrives it loads your code into one and runs it. For the simple case
there is no Dockerfile, no Service and no Ingress to write. If your code is one
function rather than an application, this is the lightest way to put it on
CIRRUS.

Python, Go, Node.js, Ruby, PHP, Bash, Java and plain binaries are supported.

**Access is a ticket.** Ask for FaaS access; you get a namespace and, if you do
not already have it, Kubernetes API access
([page 1](01-orientation.ipynb#kubectl-from-your-own-machine)). There is no GUI:
you work with the [`fission` CLI](https://fission.io/docs/installation/#install-fission-cli)
and `kubectl`, both installed on your own machine.

---

## The four pieces

| piece | what it is |
| --- | --- |
| **Environment** | the language runtime your functions plug into. One per language, shared by every function in it. |
| **Package** | your code as a deployment archive (a zip), plus its build status. Created for you when you deploy with `--src`. |
| **Function** | binds a package to an entrypoint, such as `myfunc.main`. |
| **Route** (an `HTTPTrigger`) | exposes the function at a URL on the shared gateway, `fn.k8s.ucar.edu`. |

In order — environment once, then the function, then its route:

```bash
# once per language, per namespace
fission env create --name python --poolsize=1 \
  --image ghcr.io/fission/python-env \
  --builder ghcr.io/fission/python-builder \
  -n <your-namespace>

# a single-file function with no dependencies
fission fn create --name hello --env python --code hello.py -n <your-namespace>

# expose it at https://fn.k8s.ucar.edu/<username>/hello
fission route create --function hello --name hello --url /<username>/hello \
  --route-provider gateway --gateway traefik/traefik-gateway \
  --route-host=fn.k8s.ucar.edu -n <your-namespace>
```

A function with a few pip dependencies ships a `requirements.txt` beside it and
lets Fission's builder install them. Name the module after what it does, not
after a package it imports:

```bash
zip -r src.zip myfunc.py requirements.txt
fission fn create --name myfunc --env python --src src.zip \
  --entrypoint myfunc.main -n <your-namespace>
fission pkg list -n <your-namespace>        # wait for BUILD_STATUS: succeeded
```

---

## The four things to know before you start

* **URL paths are global and first-come, first-served.** Every route on every
  namespace shares the `fn.k8s.ucar.edu` host. **Prefix your paths with your
  username, team or project** — `/<username>/hello`, `/<project>/subset`. A path
  that is already taken does not give a useful error: the existing route keeps
  working and yours shows `READY False` in `fission route list`. A route stuck at
  `False` with nothing else wrong is almost always this; delete it and choose
  another path.
* **Build status lives on the *package*, not the function.** `fission pkg list`
  shows it and `fission pkg info --name <pkg> -n <your-namespace>` has the build
  logs. A function that never becomes ready is usually a package that never
  built.
* **Environments are shared, and they cost quota.** Each keeps `poolsize` warm
  pods running against your namespace's quota whether or not anything calls it.
  One per language you actually use, reused by every function in it.
* **Bake heavy dependencies into an image.** The builder is fine for a few
  simple pip packages. A scientific stack, conda, or a shared internal library
  wants a custom runtime image instead, built once and pushed to Harbor
  ([page 2](02-containers.ipynb)). It must extend Fission's own runtime, which
  provides the function server — a plain `python` base image will not work:

  ```dockerfile
  FROM ghcr.io/fission/python-env
  COPY requirements.txt /tmp/requirements.txt
  RUN pip install --no-cache-dir -r /tmp/requirements.txt
  ```

  Point the environment at that image with no builder, and your deployment zip
  stays tiny: just your module.

---

## Test and troubleshoot

```bash
fission env list   -n <your-namespace>
fission fn list    -n <your-namespace>
fission pkg list   -n <your-namespace>
fission route list -n <your-namespace>      # READY should be True
curl -i https://fn.k8s.ucar.edu/<username>/hello
```

`curl -i` shows the status code, which is what you need when something is wrong.
A `404 page not found` means the route: check `fission route list`, and that
everything was created in your own namespace. If it all looks right and still
does not work, open a ticket with the namespace, the function name and the
`curl -i` output.

---

## Functions under GitOps

Everything above is imperative, which sits badly with
[page 5](05-argocd.ipynb). It does not have to stay that way. The CLI is the
fastest way to get a first function working; for anything you intend to keep
running, every object has a manifest equivalent that can live in git and be
reconciled by Argo CD:

| CLI | manifest |
| --- | --- |
| `fission env create` | `Environment` |
| `fission fn create --src` | `Package` |
| `fission fn create` | `Function` |
| `fission route create` | `HTTPTrigger` |

The CIRRUS layout puts the runtime image, the function code and the manifests in
one repository, with a workflow that keeps them in step:

```
.
├── image/                  Dockerfile + requirements.txt for the runtime
├── functions/myfunc.py     your function module(s)
├── manifests/              what Argo CD syncs
│   ├── environment.yaml
│   ├── package-myfunc.yaml
│   ├── function-myfunc.yaml
│   └── httptrigger-myfunc.yaml
└── .github/workflows/release.yaml
```

On every push, the release workflow builds and pushes the runtime image, zips
each function and uploads the zip as a GitHub release asset, then pins three
values into the manifests and commits them back: **the image tag, the archive
URL, and the archive checksum.** It is [page 8](08-github-actions.ipynb)'s
commit-back pattern, with `[skip ci]` and `concurrency` for the same reasons.

Two details are non-obvious, and both cost an afternoon if you meet them cold:

* **Point the `Package` at an immutable archive.** It names a zip by
  `spec.deployment.url` plus `checksum.sum`. Use a GitHub *release asset* — a
  `github.com/.../archive/...` auto-generated zip is not byte-stable, and the
  checksum will break.
* **Change the `Function` on every release.** Warm pods cache their
  specialisation, so if the `Function` spec is byte-for-byte identical after a
  release, nothing re-specialises and **the old code keeps serving**. The CIRRUS
  pattern is an annotation carrying the archive checksum, which CI bumps:

  ```yaml
  apiVersion: fission.io/v1
  kind: Function
  metadata:
    name: myfunc
    annotations:
      cirrus.ucar.edu/code-checksum: "<sha256 of this release's zip>"
  spec:
    environment:
      name: python-app
    package:
      packageref:
        name: myfunc
      functionName: myfunc.main
  ```

Kubernetes object names cannot contain underscores and Python module names
usually do: a module `my_func.py` becomes objects named `my-func`, with
`functionName: my_func.main`.

To deploy, ask for the repository to be onboarded like any other application
([page 5](05-argocd.ipynb#onboarding-an-application-on-cirrus)). If you already
have an application synced from a Helm chart, you can instead drop the
manifests into that chart's `templates/` and they sync with the rest of it.
Either way, once Argo CD owns them, `fission` and `kubectl` commands that
*modify* these objects hold only until the next sync. Reads are unaffected.

**A complete example to copy:**
[`fission/matplotlib`](https://github.com/NCAR/cirrus-examples/tree/main/fission/matplotlib)
in `cirrus-examples` is a function that fetches a compressed NetCDF file, reads
it with xarray, and returns a map as a PNG. It uses a custom runtime image,
because its scientific stack is exactly what the builder is not for.

Docs: <https://ncar-hpc-docs.readthedocs.io/en/latest/compute-systems/cirrus/guides/12-faas/faas/>
and, for the full release workflow and manifests,
<https://ncar-hpc-docs.readthedocs.io/en/latest/compute-systems/cirrus/guides/12-faas/gitops/>

---

## Check yourself

1. Your route shows `READY False` and nothing else is wrong. What is the most
   likely cause, and what do you change?
2. Your function never becomes ready. Where is the build log?
3. Why must a custom runtime image start `FROM ghcr.io/fission/python-env`
   rather than `FROM python`?
4. You deploy new function code through Argo CD and the old code keeps serving.
   Why, and what fixes it?
5. Why point a `Package` at a release asset rather than an `/archive/` zip?

---

← [11. Jupyter](11-jupyter.ipynb) · next: [13. MPI](13-mpi.ipynb)